In [ ]:
from google.colab import userdata  # keys are stored in Colab Secrets
# ==========================================
# 🤖 CELL 4: TELEGRAM BOT WITH IMAGE SUPPORT
# ==========================================

!pip install -q python-telegram-bot

from telegram import Update
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes
import requests
import base64

# --- CONFIGURATION ---
TELEGRAM_TOKEN = userdata.get('TELEGRAM_TOKEN')
API_URL = userdata.get('API_URL')  # the /ask URL printed by 2_rag_api

print("=" * 70)
print("🤖 TELEGRAM BOT WITH IMAGE SUPPORT")
print(f"📡 API: {API_URL}")
print("=" * 70)

# --- Commands ---

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    welcome = """
🏥 **Ramadan Diabetes Expert**

السلام عليكم! Welcome!

💬 **Ask me in any language** or **send a photo** of your glucose meter!

📝 **Examples:**
- "3endi 60 fssekar, wach nsom?"
- Send a photo of your blood sugar reading
- "Can I fast with Type 2 diabetes?"

⚕️ Based on IDF-DAR guidelines.
"""
    await update.message.reply_text(welcome, parse_mode='Markdown')

async def help_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text("Send /start for help!\n\n📸 You can send photos of glucose readings!")

# Handle TEXT messages
async def handle_text(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_message = update.message.text
    user_name = update.message.from_user.first_name

    await update.message.chat.send_action("typing")

    try:
        print(f"📤 {user_name}: {user_message}")

        response = requests.post(
            API_URL,
            json={"text": user_message, "image": None},
            timeout=30
        )

        if response.status_code == 200:
            answer = response.json().get("response", "⚠️ No response")
            await update.message.reply_text(answer)
            print(f"✅ Responded to {user_name}")
        else:
            error_detail = response.json().get("detail", "Unknown error")
            await update.message.reply_text(f"⚠️ Error: {error_detail}")
            print(f"❌ API error: {response.status_code} - {error_detail}")

    except Exception as e:
        await update.message.reply_text("⚠️ Error occurred")
        print(f"❌ Exception: {e}")

# Handle PHOTO messages
async def handle_photo(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_name = update.message.from_user.first_name

    await update.message.chat.send_action("typing")

    try:
        print(f"📸 {user_name} sent a photo")

        # Get the largest photo
        photo = update.message.photo[-1]

        # Download photo
        photo_file = await photo.get_file()
        photo_bytes = await photo_file.download_as_bytearray()

        # Convert to base64
        image_b64 = base64.b64encode(photo_bytes).decode('utf-8')

        # Get caption if any
        caption = update.message.caption or "Analyze this glucose reading"

        print(f"📤 Sending photo to API...")

        response = requests.post(
            API_URL,
            json={"text": caption, "image": image_b64},
            timeout=30
        )

        if response.status_code == 200:
            answer = response.json().get("response", "⚠️ No response")
            await update.message.reply_text(answer)
            print(f"✅ Photo analyzed for {user_name}")
        else:
            error_detail = response.json().get("detail", "Unknown")
            await update.message.reply_text(f"⚠️ Couldn't analyze photo: {error_detail}")
            print(f"❌ Photo error: {response.status_code}")

    except Exception as e:
        await update.message.reply_text("⚠️ Error analyzing photo")
        print(f"❌ Photo exception: {e}")

async def main():
    app = Application.builder().token(TELEGRAM_TOKEN).build()

    app.add_handler(CommandHandler("start", start))
    app.add_handler(CommandHandler("help", help_command))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_text))
    app.add_handler(MessageHandler(filters.PHOTO, handle_photo))

    print("\n" + "=" * 70)
    print("✅ BOT RUNNING WITH IMAGE SUPPORT!")
    print("📱 Go to Telegram!")
    print("📸 Send text OR photos!")
    print("=" * 70)

    await app.run_polling(allowed_updates=Update.ALL_TYPES)

import nest_asyncio
nest_asyncio.apply()
await main()
